# Your first project: test MedGemma on 20 real chest X-rays

**Goal:** find out how well a medical AI model answers one simple question about real chest X-rays, without training anything.

**The question:** *"Is there fluid around the lungs (a pleural effusion) in this X-ray?"*

You will:

1. **Get the data:** download about 5,000 chest X-rays from the NIH (free, no sign-up).
2. **Build a test set:** pick 10 X-rays that have fluid and 10 that are healthy.
3. **Ask the model:** show each X-ray to MedGemma and ask "yes or no?"
4. **Score it:** count how many it caught, missed, or raised a false alarm on.

### Before you start (5 minutes, once)

1. **Turn on a free GPU:** in the menu, choose **Runtime → Change runtime type → T4 GPU → Save**.
2. **Get access to MedGemma:** make a free account at [huggingface.co](https://huggingface.co), open [google/medgemma-4b-it](https://huggingface.co/google/medgemma-4b-it), and accept the terms.
3. **Make an access token:** on Hugging Face go to **Settings → Access Tokens → Create new token** (type "Read"). You'll paste it below.

Then run each cell in order with **Shift + Enter**.

In [ ]:
# The plan, as a picture. Also checks that the GPU is on.
from IPython.display import SVG, display
import torch

steps = [("1. Get the data", "NIH ChestX-ray14"), ("2. Build a test set", "10 fluid, 10 healthy"),
         ("3. Ask the model", "MedGemma: yes or no?"), ("4. Score it", "caught, missed, false alarms")]
boxes = ""
for i, (name, sub) in enumerate(steps):
    x = 10 + i * 190
    fill = "#e7f1fb" if i == 2 else "white"
    boxes += (f'<rect x="{x}" y="10" width="170" height="60" rx="8" fill="{fill}" stroke="#888"/>'
              f'<text x="{x + 85}" y="36" text-anchor="middle" font-weight="bold" font-size="14">{name}</text>'
              f'<text x="{x + 85}" y="56" text-anchor="middle" font-size="12" fill="#555">{sub}</text>')
    if i < 3:
        boxes += f'<path d="M{x + 172} 40 H{x + 188}" stroke="#888" marker-end="url(#a)"/>'
display(SVG(f'<svg xmlns="http://www.w3.org/2000/svg" width="780" height="80" font-family="sans-serif">'
            f'<defs><marker id="a" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto">'
            f'<path d="M0 0L10 5L0 10z" fill="#888"/></marker></defs>{boxes}</svg>'))

if torch.cuda.is_available():
    print("GPU is on:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Go to Runtime -> Change runtime type -> T4 GPU, then run this cell again.")

## Setup: install the tools and log in to Hugging Face

This takes a minute or two. When the login box appears, paste your access token.

In [ ]:
!pip install -q -U "transformers>=4.50.0" accelerate

from huggingface_hub import notebook_login
notebook_login()

## Step 1: get the data

We download the first batch of **ChestX-ray14**, a free dataset from the U.S. National Institutes of Health (NIH): about 5,000 chest X-rays (around 2 GB), plus a label file saying what doctors' reports found in each one.

Source: https://nihcc.app.box.com/v/ChestXray-NIHCC. This takes a few minutes.

In [ ]:
import os, tarfile, urllib.request
from pathlib import Path

DATA = Path("data")
DATA.mkdir(exist_ok=True)

# Direct download links for the NIH files (the same links the open-source PyHealth library uses)
LABELS_URL = ("https://nihcc.app.box.com/index.php?rm=box_download_shared_file"
              "&vanity_name=ChestXray-NIHCC&file_id=f_219760887468")
IMAGES_URL = "https://nihcc.box.com/shared/static/vfk49d74nhbxq3nqjg0900w5nvkorp5c.gz"  # images_001

labels_file = DATA / "Data_Entry_2017_v2020.csv"
if not labels_file.exists():
    print("Downloading the label file ...")
    urllib.request.urlretrieve(LABELS_URL, labels_file)

images_dir = DATA / "images"
if not images_dir.exists():
    tar_path = DATA / "images_001.tar.gz"
    print("Downloading about 2 GB of X-rays (a few minutes) ...")
    urllib.request.urlretrieve(IMAGES_URL, tar_path)
    print("Unpacking ...")
    with tarfile.open(tar_path) as tar:
        try:
            tar.extractall(DATA, filter="data")   # safe extraction (newer Python)
        except TypeError:
            tar.extractall(DATA)                  # older Python without the filter option
    tar_path.unlink()  # free up disk space

print("X-rays downloaded:", len(list(images_dir.glob("*.png"))))

## Step 2: build a small test set

We pick **10 X-rays labeled "Effusion"** (fluid around the lungs) and **10 labeled "No Finding"** (healthy). A fixed random seed means everyone running this notebook gets the same 20 images, so you can compare results with classmates.

Then we look at them. Can *you* spot the fluid?

In [ ]:
import csv, random
import matplotlib.pyplot as plt
from PIL import Image

FINDING = "Effusion"   # try "Cardiomegaly" or "Pneumothorax" later
N = 10                 # images per group

downloaded = {p.name for p in images_dir.glob("*.png")}
yes_imgs, no_imgs = [], []
with open(labels_file, newline="") as f:
    for row in csv.DictReader(f):
        if row["Image Index"] not in downloaded:
            continue
        found = row["Finding Labels"].split("|")
        if FINDING in found:
            yes_imgs.append(row["Image Index"])
        elif found == ["No Finding"]:
            no_imgs.append(row["Image Index"])

print(f"Available: {len(yes_imgs)} with {FINDING}, {len(no_imgs)} with No Finding")

random.seed(42)
test_set = [(name, "yes") for name in random.sample(yes_imgs, N)] + \
           [(name, "no") for name in random.sample(no_imgs, N)]
random.shuffle(test_set)

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for ax, (name, truth) in zip(axes.flat, test_set):
    ax.imshow(Image.open(images_dir / name), cmap="gray")
    ax.set_title(f"{name}\nreal answer: {truth}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Step 3: ask the model

We load **MedGemma 4B**, Google's medical vision-language model, and ask it the same yes/no question about each X-ray. Loading takes a few minutes the first time. Then each image takes a few seconds.

The wording of the question matters. Changing it is step 2 of the "try first, train later" ladder: better instructions.

In [ ]:
from transformers import pipeline

# bfloat16 is best; the free T4 GPU doesn't support it, so we fall back to float16 there.
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
pipe = pipeline("image-text-to-text", model="google/medgemma-4b-it", torch_dtype=dtype, device="cuda")

QUESTION = ("Is there a pleural effusion (fluid around the lungs) in this chest X-ray? "
            "Answer with only one word: yes or no.")

def to_yes_no(text):
    words = text.strip().lower().split()
    first = words[0].strip(".,!:;*\"'") if words else ""
    return first if first in ("yes", "no") else "unclear"

results = []
for i, (name, truth) in enumerate(test_set, start=1):
    image = Image.open(images_dir / name).convert("RGB")
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are an expert radiologist."}]},
        {"role": "user", "content": [{"type": "text", "text": QUESTION}, {"type": "image", "image": image}]},
    ]
    reply = pipe(text=messages, max_new_tokens=20)[0]["generated_text"][-1]["content"].strip()
    answer = to_yes_no(reply)
    results.append({"image": name, "truth": truth, "answer": answer, "reply": reply})
    mark = "right" if answer == truth else "WRONG"
    print(f"[{i:2}/20] {name}  real: {truth:3}  model: {answer:7}  {mark}   (said: {reply!r})")

## Step 4: score it

In medicine, the *kind* of mistake matters:

- **Missed it:** the fluid was there, but the model said no. A sick patient could be sent home.
- **False alarm:** the lungs were healthy, but the model said yes. A patient could get worried or have extra tests.

In [ ]:
caught      = sum(r["truth"] == "yes" and r["answer"] == "yes" for r in results)
missed      = sum(r["truth"] == "yes" and r["answer"] != "yes" for r in results)
false_alarm = sum(r["truth"] == "no"  and r["answer"] == "yes" for r in results)
cleared     = sum(r["truth"] == "no"  and r["answer"] == "no"  for r in results)
unclear     = sum(r["answer"] == "unclear" for r in results)

print(f"{'':14}{'model: yes':>12}{'model: no/unclear':>20}")
print(f"{'really yes':14}{caught:>12}{missed:>20}    caught / MISSED")
print(f"{'really no':14}{false_alarm:>12}{N - false_alarm:>20}    FALSE ALARM / cleared")
print()
print(f"Accuracy:    {(caught + cleared) / len(results):.0%}   ({caught + cleared} of {len(results)} right)")
print(f"Sensitivity: {caught / N:.0%}   (real cases caught: {caught} of {N})")
print(f"Specificity: {cleared / N:.0%}   (healthy lungs cleared: {cleared} of {N})")
if unclear:
    print(f"Unclear:     {unclear} answers weren't a clear yes or no; they count as wrong.")

# Look at the mistakes: seeing HOW a model is wrong teaches you the most
wrong = [r for r in results if r["answer"] != r["truth"]]
if wrong:
    fig, axes = plt.subplots(1, len(wrong), figsize=(3 * len(wrong), 3.5), squeeze=False)
    for ax, r in zip(axes[0], wrong):
        ax.imshow(Image.open(images_dir / r["image"]), cmap="gray")
        kind = "MISSED" if r["truth"] == "yes" else "FALSE ALARM"
        ax.set_title(f"{kind}\nmodel said: {r['reply'][:20]}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## What to try next

Change **one thing at a time**, rerun Steps 2 to 4, and write down the new score each time:

1. **Reword the question** in Step 3. Does "Answer yes or no" beat "Is there any abnormality"?
2. **Ask about a different finding:** set `FINDING = "Cardiomegaly"` (enlarged heart) in Step 2 and change the question to match.
3. **Use more images:** set `N = 50`. Do the numbers change much? (With only 20 images, one image moves the score by 5 percentage points.)
4. **Ask for reasoning first:** ask the model to describe the X-ray, then answer yes or no. Does that help?

### Honest limits

- ChestX-ray14 labels were pulled out of reports by a computer program, so some are wrong. Your score shows agreement with those labels, not with the truth.
- 20 images is a demo, not a research result.
- MedGemma is a research tool, not a doctor.

### Credit

Data: NIH Clinical Center, ChestX-ray14 (https://nihcc.app.box.com/v/ChestXray-NIHCC). If you share results, cite: Wang X. et al., "ChestX-ray8: Hospital-scale Chest X-ray Database and Benchmarks," CVPR 2017.